### Notebook to look at individual researchers' corpus  

- Match Domingo's T/C researchers to OA author_ids  


- Extract the full corpus of researchers

In [13]:
import duckdb
import pandas as pd
from pathlib import Path
import diskcache
from itertools import chain

import numpy as np
from utils.pandas_setup import pandas_setup
pandas_setup()

import contextlib
from unidecode import unidecode
from nameparser import HumanName

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/econommicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')

def normalise_name(in_name: str=None) -> dict:
    # print(f'{in_name = }')
    in_name = ' '.join([part.strip() for part in unidecode(in_name).split(' ')])
    in_name = in_name.title()
    name = HumanName(in_name)
    if name.middle == "":
        fullname = f'{name.first} {name.last}'
    else:
        fullname = f'{name.first} {name.middle} {name.last}'
    return {'first': name.first, 'middle': name.middle, 'family': name.last, 'fullname': fullname}

In [14]:
class SetUp:

    def __init__(self):
        self._setup_db()
        return

    def _setup_db(self):
        self.db = duckdb.connect(MY_DATABASE_FILE)
        self.db.sql("ATTACH IF NOT EXISTS ':memory:'")
        self.db.sql(""" SET memory_limit = '56GB';
                        SET threads = 6;
                        SET preserve_insertion_order = false;
                        SET order_by_non_integer_literal=true;
                        SET enable_progress_bar = true;
                        SET temp_directory = '/home/lc/m/.tmp';
                    """)
    
        with contextlib.suppress(Exception):
            self.db.create_function('normalise_name', 
                                        normalise_name, 
                                        return_type=duckdb.duckdb.typing.DuckDBPyType(str), 
                                        exception_handling='return_null',
                                        null_handling='special',
                                        side_effects=True
                                    )
            # self.db.create_function('extract_works', 
            #                             extract_works, 
            #                             return_type=duckdb.duckdb.typing.DuckDBPyType(dict[str, str]), 
            #                         )
        self.db.sql("SHOW ALL TABLES").show()
        return

#### This cell extracts the endogenous HCRs from works in the Journal Set  

- Construct time-series of citations from reference lists  
- Compute the centile for each publication year
- Filter highly cited papers  
- Group the authors of hte highly cited papers

In [15]:
class CorpusETL(SetUp):

    def __init__(self):
        super().__init__()
        return

    def citations_per_work(self):
        sql = """ 
        CREATE OR REPLACE TABLE memory.citations_per_work AS
        SELECT c.cited_id,
                count(c.citer_id) AS cited_by_count_endogenous,
                w.cited_by_count AS cited_by_count_total,
                w.publication_year
        FROM
        (SELECT work_id AS citer_id,
                unnest(referenced_works) AS cited_id
            FROM cited
            ) c
        LEFT JOIN works w
        ON c.cited_id = w.work_id
        WHERE w.work_id NOT NULL
        GROUP BY ALL
        ORDER BY publication_year, cited_by_count_endogenous DESC, cited_by_count_total DESC
        """
        self.db.sql(sql)
        return

    def citations_per_work_ranked(self):
        sql = """ 
                CREATE OR REPLACE TABLE memory.citations_per_work_ranked AS
                SELECT cited_id,
                        publication_year,
                        cited_by_count_total,
                        cited_by_count_endogenous,
                        percent_rank(ORDER BY cited_by_count_total) OVER w AS percent_rank_total,
                        percent_rank(ORDER BY cited_by_count_endogenous) OVER w AS percent_rank_endogenous
                FROM memory.citations_per_work
                WINDOW w AS (PARTITION BY publication_year) -- ORDER BY cited_by_count_total, cited_by_count_endogenous) 
                ORDER BY publication_year DESC, percent_rank_total DESC
                """
        self.db.sql(sql)
        return

    def citation_summation(self):
        sql = """ 
                CREATE OR REPLACE TABLE memory.citations AS
                SELECT author_id,
                        author_name,
                        sum(cited_by_count_total) AS citations_total,
                        sum(cited_by_count_endogenous) AS citations_endogenous
                FROM memory.citations_per_work_ranked m
                LEFT JOIN authorships a
                ON m.cited_id = a.work_id
                WHERE author_id NOT NULL
                GROUP BY author_id, author_name
                """
        self.db.sql(sql)
        return

    def hca_summation(self):       

        sql = """ 
                CREATE OR REPLACE TABLE memory.hca_endogenous AS
                SELECT author_id,
                        author_name,
                        count(cited_id) AS hca_endogenous
                FROM memory.citations_per_work_ranked m
                LEFT JOIN authorships a
                ON m.cited_id = a.work_id
                WHERE a.work_id NOT NULL 
                        AND percent_rank_endogenous >= 0.99
                GROUP BY ALL
                ORDER BY hca_endogenous DESC;

                CREATE OR REPLACE TABLE memory.hca_total AS
                SELECT author_id,
                        author_name,
                        count(cited_id) AS hca_total,
                FROM memory.citations_per_work_ranked m
                LEFT JOIN authorships a
                ON m.cited_id = a.work_id
                WHERE a.work_id NOT NULL 
                        AND percent_rank_total >= 0.99
                GROUP BY ALL
                ORDER BY hca_total DESC
                """
        self.db.sql(sql)
        return
    
    def author_works_count(self):
        sql = """
            CREATE OR REPLACE TABLE econ.author_works_counts AS
            SELECT DISTINCT author_id,
                    author_name,
                    count(DISTINCT work_id) AS works_count_endogenous,
                    aus.*
            FROM econ.authorships a
            LEFT JOIN econ.authors aus
            USING (author_id, author_name)
            WHERE author_id NOT NULL
            GROUP BY ALL
            ORDER BY works_count_endogenous DESC
            """
        self.db.sql(sql)
        return
    
    def citation_summary(self):

        sql = """ 
            CREATE OR REPLACE TABLE econ.citation_summary AS
            SELECT author_id,
                    author_name,
                    citations_total,
                    citations_endogenous,
                    hca_total,
                    hca_endogenous,
                    works_count_endogenous,
                    au.*
            FROM memory.citations
            LEFT JOIN
                (SELECT t.*,
                        e.hca_endogenous,
                FROM memory.hca_total t
                LEFT JOIN memory.hca_endogenous e
                USING (author_id)
                ) sub
                USING (author_id, author_name)
                LEFT JOIN econ.author_works_counts au
                USING (author_id, author_name)
            ORDER BY citations_total DESC
            """
        self.db.sql(sql)
        return
    
    def show_all(self):
        self.db.sql("SELECT * FROM memory.citations_per_work").show()
        self.db.sql("SELECT * FROM memory.citations_per_work_ranked").show()
        self.db.sql("SELECT * FROM memory.citations").show()
        self.db.sql("SELECT * FROM memory.hca_endogenous").show() 
        self.db.sql("SELECT * FROM econ.authors").show()  
        self.db.sql("SELECT * FROM econ.citation_summary").show()
        return
    
    def load_citations(self):
        df = self.db.sql("SELECT * FROM econ.citation_summary ORDER BY citations_total DESC").df().reset_index(drop=True)
        df.to_excel('../DATA/citation_summary.xlsx')
        return


#### This cell matches Domingo's C and T lists to authors in the OpenAlex extract from the Journal Set  

- Extract Domingo's list and ensure that the names are normalised

- Compare with OpenAlex lists  

    - HCRs - endogenous - from OpenAlex references in journal set  
    - Authorships - endogenous - from OpenAlex works in journal set   
    - Authors - exogenous - from the entire OpenAlex author dataest, filtered into eeconomics and Business topics

In [16]:
    
class MatchDomingoSample(SetUp):

    def __init__(self):
        super().__init__()
        return    

    def extract_sample(self):
        sample = pd.read_excel('../RESULTS/researchers_results.xlsx').drop(columns=['Unnamed: 0', 'NAME'])
        print(f'{sample.shape = }\n{sample.head()}')
        sample['Research_Profile'] = [normalise_name(n).get('fullname') for n in sample.Research_Profile]
        sample['first'] = [normalise_name(n).get('first') for n in sample.Research_Profile]
        sample['last'] = [normalise_name(n).get('family') for n in sample.Research_Profile]
        sample['found'] = False
        sample = sample.sort_values('HCP', ascending=False).reset_index(drop=True)
        print('Duplicate names in Domingo list?')
        print(sample[sample.duplicated(keep=False)].head())
        self.db.sql("CREATE OR REPLACE TABLE econ.domingo_sample AS SELECT * FROM sample")
        self.sample = self.db.sql("SELECT * FROM econ.domingo_sample").df()
        print(f'{sample.shape = }\n{sample.head()}')
        return
    
    def compare_sample_hca(self):
        self.hca = self.db.sql("SELECT * FROM econ.hcp_count_endogenous").df()
        self.hca = self.hca[self.hca.hcp_count > 3]
        self.sample_test = self.sample[self.sample['Group'] == 'T']
        print(f'{self.hca.shape = }\n{self.hca.head(128)}')
        print(f'{self.sample_test.shape = }\n{self.sample_test.head(128)}')
        for row in self.sample_test.itertuples():
            found = False
            for row1 in self.hca.itertuples():
                if row.Research_Profile == row1.author_name:
                    print(f'MATCH full {row.Index} {row.Research_Profile = }')
                    self.sample.at[row.Index, 'found'] = True
                    found = True
                    continue
        self._compare_sample_hca()
        print(f'{self.sample_test.shape = }\n{self.sample_test.head(32)}')
        print(f'{self.sample_test[self.sample_test.found].shape = }\n{self.sample_test[self.sample_test.found].head(16)}')
        print(f'{self.sample_test[~self.sample_test.found].shape = }\n{self.sample_test[~self.sample_test.found].head(16)}')
        return
    
    def _compare_sample_hca(self):
        for row in self.sample_test.itertuples():
            if row.found:
                continue
            found = False
            for row1 in self.hca.itertuples():
                if row1.hcp_count < 3:
                    continue
                if row.last.lower() == row1.family.lower() and row.first[0] == row1.first[0]:
                    print(f'match LAST {row.Index} {row.Research_Profile = }')
                    self.sample_test.at[row.Index, 'found'] = True
                    found = True
                    continue
        return                          
         
    
    def match_sample_fullname(self):
        sql = """ 
            SELECT Research_Profile,
                    count(Research_Profile) AS match_count_domingo,
                    count(DISTINCT author_id) AS match_count_openalex,
                    author_id,
                    author_name,
                    "Group", 
                    CIT,
                    PUB,
                    HCP,
                    h_index,
                    s.first,
                    s.last,
                    works_count,
                    array_to_string(list(DISTINCT country_code), ' ') AS country_codes,
                    array_to_string(list(DISTINCT institution_name), ' ') AS institution_names,
                FROM econ.domingo_sample s
                    LEFT JOIN econ.authors a
                        ON lower(a.author_name) = lower(s.Research_Profile) 
                            -- OR list_contains(str_split(a.author_name, ' '), last) = true
                    --WHERE a.author_id NOT NULL
                    GROUP BY ALL
                    ORDER BY s.last ASC, works_count DESC
            """
        df = self.db.sql(sql).df()
        df.to_csv('../DATA/domingo_sample_full_name_match.csv')
        df = df.drop_duplicates() #subset='last')
        print(f'>>fullname match {df.shape = }\n{df.head()}')
        # self.db.sql("CREATE OR REPLACE TABLE econ.sample_names AS SELECT * FROM df")
        return
    
    def match_sample_lastname(self):
        sql = """ 
            SELECT Research_Profile,
                    author_id,
                    author_name,
                    "Group", 
                    CIT,
                    PUB,
                    HCP,
                    h_index,
                    s.first,
                    s.last,
                    works_count,
                    array_to_string(list(country_code), ' ') AS country_codes,
                    array_to_string(list(institution_name), ' ') AS institution_names,
                FROM econ.domingo_sample s
                    LEFT JOIN econ.authors a
                        ON list_contains(str_split(a.author_name, ' '), s.last) = true 
                            AND author_name[1] = Research_Profile[1]
                    WHERE a.author_id NOT NULL
                    GROUP BY ALL
                    ORDER BY s.last ASC, works_count DESC
            """
        df = self.db.sql(sql).df()
        df.to_csv('../DATA/domingo_sample_last_name_match.csv')
        df = df.drop_duplicates()  #subset='last')
        print(f'last name match{df.shape = }\n{df.head()}')
        # self.db.sql("CREATE OR REPLACE TABLE econ.sample_names AS SELECT * FROM df")
        return


In [17]:
def main():

    cetl = CorpusETL()
    cetl.citations_per_work()
    cetl.citations_per_work_ranked()
    cetl.citation_summation()
    cetl.hca_summation()
    cetl.author_works_count()
    cetl.citation_summary()
    cetl.show_all()
    cetl.load_citations()

    # mds = MatchDomingoSample()
    # mds.extract_sample()
    # mds.compare_sample_hca()
    # mds.match_sample_fullname()
    # mds.match_sample_lastname()


        
    return

In [18]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬───────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────